In [1]:
# 1. Project/config — chỉ điều phối, business logic nằm trong src/data_pipeline.py
from pathlib import Path
import os, sys
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / 'config' / 'config.yaml').is_file())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.data_pipeline import DataPreparationPipeline
pipeline = DataPreparationPipeline(PROJECT_ROOT)
pipeline.config

{'project': {'name': 'weather_disease_ai_v3', 'random_seed': 42},
 'timezone': 'Asia/Ho_Chi_Minh',
 'data': {'patient_file': 'data/raw/train_history.xlsx',
  'weather_file': 'data/raw/weather_hcm_history.csv',
  'original_patient_source': '../seasonal_disease_backend/Tool/weather/train_history.xlsx',
  'original_weather_source': '../seasonal_disease_backend/Tool/weather/open-meteo-10.79N106.63E6m.csv',
  'patient_sheet': 'DS-BenhNhan',
  'disease_sheet': 'DS-MaBenh',
  'output_format': 'csv.gz'},
 'weather_lookback': {'current': True, 'days': [3, 7]},
 'prediction_horizons': [3, 7, 14],
 'target_includes_anchor_day': True,
 'split': {'train_ratio': 0.7,
  'validation_ratio': 0.15,
  'test_ratio': 0.15,
  'purge_gap_days': 13,
  'split_unit': 'unique_anchor_date'},
 'support': {'definition': 'summed_horizon_case_count_on_train_queries',
  'high_min': 1000,
  'medium_min': 200,
  'low_min': 20}}

In [2]:
# 2. Load raw data
source_info = pipeline.load_raw_data()
source_info

{'original_patient_source': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\seasonal_disease_backend\\Tool\\weather\\train_history.xlsx',
 'original_weather_source': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\seasonal_disease_backend\\Tool\\weather\\open-meteo-10.79N106.63E6m.csv',
 'v3_patient_raw_copy': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\weather_disease_ai_v3\\data\\raw\\train_history.xlsx',
 'v3_weather_raw_copy': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\weather_disease_ai_v3\\data\\raw\\weather_hcm_history.csv',
 'patient_sha256': 'ae148fdadbcc901a3ade4bff9281bc856f34fc24906fa0c561a2cb09a3f28c78',
 'weather_sha256': 'd40d4931966e89778a959598ec5b4ecb701f8f67b0ae3356d89c6c164797ae82',
 'workbook_sheets': ['DS-BenhNhan', 'DS-MaBenh'],
 'raw_patient_rows': 257771,
 'raw_catalog_rows': 12230,
 'weather_hourly_rows': 37944}

In [3]:
# 3. Audit source data
source_audit = pipeline.audit_source_data()
source_audit

{'original_patient_source': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\seasonal_disease_backend\\Tool\\weather\\train_history.xlsx',
 'original_weather_source': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\seasonal_disease_backend\\Tool\\weather\\open-meteo-10.79N106.63E6m.csv',
 'v3_patient_raw_copy': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\weather_disease_ai_v3\\data\\raw\\train_history.xlsx',
 'v3_weather_raw_copy': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\weather_disease_ai_v3\\data\\raw\\weather_hcm_history.csv',
 'patient_sha256': 'ae148fdadbcc901a3ade4bff9281bc856f34fc24906fa0c561a2cb09a3f28c78',
 'weather_sha256': 'd40d4931966e89778a959598ec5b4ecb701f8f67b0ae3356d89c6c164797ae82',
 'workbook_sheets': ['DS-BenhNhan', 'DS-MaBenh'],
 'raw_patient_rows': 257771,
 'raw_catalog_rows': 12230,
 'weather_hourly_rows': 37944,
 'weather_actual_columns': ['time',
  'temperature_2m (°C

In [4]:
# 4. Build daily disease counts
daily_cases = pipeline.build_daily_disease_counts()
daily_cases.head()

,date,age_group,gender,disease_group_id,disease_group_name,case_count_day
0,2021-01-01,1-5 tuổi,Nam,169,Các bệnh viêm phổi - Pneumonia,2
1,2021-01-01,1-5 tuổi,Nam,198,Bệnh nhiễm khuẩn da và mô tế bào dưới da - Inf...,1
2,2021-01-01,Dưới 1 tuổi,Nam,165,Viêm họng và viêm amidan cấp - Acute pharyngit...,1
3,2021-01-02,1-5 tuổi,Nam,169,Các bệnh viêm phổi - Pneumonia,1
4,2021-01-02,1-5 tuổi,Nữ,170,Viêm phế quản và viêm tiểu phế quản cấp - Acut...,1


In [5]:
# 5. Weather current + exact trailing 3d/7d
weather_features = pipeline.build_weather_current_3d_7d()
weather_features.dropna().head()

,anchor_date,weather_code_current,temperature_mean_current,temperature_max_current,temperature_min_current,humidity_mean_current,humidity_max_current,humidity_min_current,wind_speed_mean_current,wind_speed_max_current,...,humidity_mean_7d,humidity_max_7d,humidity_min_7d,wind_speed_mean_7d,wind_speed_max_7d,precipitation_sum_7d,rain_sum_7d,rain_days_7d,rain_max_daily_7d,wind_gust_max_7d
6,2021-01-07,0.0,28.670833,35.2,24.5,61.250000,78,32,7.441667,19.8,...,60.678571,84.0,32.0,7.669643,21.6,5.5,5.5,5.0,3.9,49.0
7,2021-01-08,3.0,28.300000,34.5,23.4,60.875000,83,34,7.245833,17.6,...,61.898810,84.0,32.0,7.044643,19.8,5.5,5.5,5.0,3.9,49.0
8,2021-01-09,2.0,27.091667,32.5,22.6,57.666667,71,37,8.275000,18.2,...,62.166667,84.0,32.0,7.180952,19.8,5.2,5.2,4.0,3.9,49.0
9,2021-01-10,3.0,27.016667,32.9,21.9,58.208333,72,34,6.554167,18.9,...,61.791667,84.0,32.0,7.205952,19.8,5.0,5.0,3.0,3.9,49.0
10,2021-01-11,3.0,27.554167,34.3,21.4,56.333333,77,32,7.870833,24.4,...,60.666667,84.0,32.0,7.375000,24.4,4.0,4.0,2.0,3.9,49.0


In [6]:
# 6. Context/query dataset
contexts = pipeline.build_query_contexts()
contexts.head()

,query_id,anchor_date,age_group,gender,month,season,day_of_year_sin,day_of_year_cos,weather_code_current,temperature_mean_current,...,humidity_mean_7d,humidity_max_7d,humidity_min_7d,wind_speed_mean_7d,wind_speed_max_7d,precipitation_sum_7d,rain_sum_7d,rain_days_7d,rain_max_daily_7d,wind_gust_max_7d
0,Q20210107_C00,2021-01-07,1-5 tuổi,Nam,1,Mùa khô,0.120126,0.992759,0.0,28.670833,...,60.678571,84.0,32.0,7.669643,21.6,5.5,5.5,5.0,3.9,49.0
1,Q20210107_C01,2021-01-07,1-5 tuổi,Nữ,1,Mùa khô,0.120126,0.992759,0.0,28.670833,...,60.678571,84.0,32.0,7.669643,21.6,5.5,5.5,5.0,3.9,49.0
2,Q20210107_C02,2021-01-07,11-15 tuổi,Nam,1,Mùa khô,0.120126,0.992759,0.0,28.670833,...,60.678571,84.0,32.0,7.669643,21.6,5.5,5.5,5.0,3.9,49.0
3,Q20210107_C03,2021-01-07,11-15 tuổi,Nữ,1,Mùa khô,0.120126,0.992759,0.0,28.670833,...,60.678571,84.0,32.0,7.669643,21.6,5.5,5.5,5.0,3.9,49.0
4,Q20210107_C04,2021-01-07,6-10 tuổi,Nam,1,Mùa khô,0.120126,0.992759,0.0,28.670833,...,60.678571,84.0,32.0,7.669643,21.6,5.5,5.5,5.0,3.9,49.0


In [7]:
# 7. Sparse targets for H3/H7/H14
targets = pipeline.build_horizon_targets()
targets.head()

,query_id,disease_group_id,case_count_h3,has_case_h3,case_count_h7,has_case_h7,case_count_h14,has_case_h14
0,Q20210107_C00,165,0,0,4,1,4,1
1,Q20210107_C00,169,5,1,12,1,19,1
2,Q20210107_C00,170,2,1,5,1,8,1
3,Q20210107_C00,176,0,0,2,1,3,1
4,Q20210107_C00,182,1,1,1,1,1,1


In [8]:
# 8. Chronological split with H14 purge
split_frames = pipeline.build_temporal_splits()
pipeline.split_metadata

{'method': 'chronological_unique_anchor_dates_70_15_15_with_h14_purge',
 'purge_gap_days': 13,
 'max_horizon_days': 14,
 'train': {'date_from': '2021-01-07',
  'date_to': '2023-12-14',
  'unique_anchor_dates': 1072,
  'queries': 12864},
 'validation': {'date_from': '2023-12-28',
  'date_to': '2024-08-12',
  'unique_anchor_dates': 229,
  'queries': 2748},
 'test': {'date_from': '2024-08-26',
  'date_to': '2025-04-13',
  'unique_anchor_dates': 231,
  'queries': 2772},
 'purge_train_validation': {'date_from': '2023-12-15',
  'date_to': '2023-12-27',
  'unique_anchor_dates': 13,
  'queries_excluded': 156},
 'purge_validation_test': {'date_from': '2024-08-13',
  'date_to': '2024-08-25',
  'unique_anchor_dates': 13,
  'queries_excluded': 156},
 'checks': {'no_anchor_overlap': True,
  'train_h14_before_validation': True,
  'validation_h14_before_test': True}}

In [9]:
# 9. Train-only disease support
disease_catalog = pipeline.calculate_train_support()
pipeline.support_counts

{'h3': {'high': 65,
  'medium': 43,
  'low': 70,
  'insufficient': 43,
  'unsupported': 90},
 'h7': {'high': 85,
  'medium': 57,
  'low': 56,
  'insufficient': 23,
  'unsupported': 90},
 'h14': {'high': 104,
  'medium': 55,
  'low': 44,
  'insufficient': 18,
  'unsupported': 90}}

In [10]:
# 10. Validation/assertions
checks = pipeline.validate_all()
assert all(checks.values())
checks

{'no_duplicate_query_id': True,
 'no_duplicate_target_key': True,
 'weather_current_is_d_only': True,
 'weather_3d_is_d_minus_2_to_d': True,
 'weather_7d_is_d_minus_6_to_d': True,
 'no_future_weather': True,
 'h3_is_d_to_d_plus_2': True,
 'h7_is_d_to_d_plus_6': True,
 'h14_is_d_to_d_plus_13': True,
 'h3_le_h7_le_h14': True,
 'has_case_matches_case_count': True,
 'target_windows_within_source': True,
 'no_anchor_overlap': True,
 'no_target_window_overlap_between_splits': True,
 'disease_catalog_mapping_valid': True,
 'context_has_no_disease_target': True,
 'anchor_date_excluded_from_model_features': True,
 'year_excluded_from_model_features': True,
 'support_built_from_train_only': True,
 'baseline_contract_train_only': True,
 'same_weather_features_for_h3_h7_h14': True}

In [11]:
# 11. Three real-date examples
examples = pipeline.build_examples()
examples

[{'query_id': 'Q20210107_C00',
  'anchor_date': '2021-01-07',
  'age_group': '1-5 tuổi',
  'gender': 'Nam',
  'weather_ranges': {'current': '2021-01-07',
   '3d': '2021-01-05 -> 2021-01-07',
   '7d': '2021-01-01 -> 2021-01-07'},
  'target_ranges': {'h3': '2021-01-07 -> 2021-01-09',
   'h7': '2021-01-07 -> 2021-01-13',
   'h14': '2021-01-07 -> 2021-01-20'},
  'weather_values': {'temperature_mean_current': 28.670833,
   'humidity_mean_current': 61.25,
   'precipitation_sum_3d': 4.0,
   'wind_speed_mean_7d': 7.669643},
  'positive_h3': [{'disease_group_id': '169',
    'disease_group_name': 'Các bệnh viêm phổi - Pneumonia',
    'case_count': 5},
   {'disease_group_id': '170',
    'disease_group_name': 'Viêm phế quản và viêm tiểu phế quản cấp - Acute bronchitis and acute bronchiolitis',
    'case_count': 2},
   {'disease_group_id': '182',
    'disease_group_name': 'Bệnh khác của khoang miệng, tuyến nước bọt và hàm - Other diseases of the oral cavity, salivary glands and jaws',
    'case_cou

In [12]:
# 12. Save artifacts and preparation report — no model training
metadata = pipeline.save_artifacts()
assert metadata['no_model_trained'] is True
{'queries': metadata['contexts']['queries'], 'report': str(PROJECT_ROOT / 'reports' / 'DATA_PREPARATION_REPORT.md'), 'model_trained': False}

{'queries': 18696,
 'report': 'D:\\OS_C\\Bài học trên trường\\Thực tập\\Fullstack_2\\ThucTap-main\\weather_disease_ai_v3\\reports\\DATA_PREPARATION_REPORT.md',
 'model_trained': False}